# 02 — Failure Labels & Early-Warning Baseline

**Run this notebook top-to-bottom.** It is intentionally self-contained: it does not depend on Python variables created in another notebook. It rebuilds the FFIEC bank-quarter panel, matches FDIC failures, creates the next-4-quarter target, and evaluates the baseline ML models.

## 1. Imports and project paths

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/Users/irfanhussain/Desktop/systemic-risk-project")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FFIEC_DIR = RAW_DIR / "ffiec"
FDIC_DIR = RAW_DIR / "fdic"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert FFIEC_DIR.exists(), f"Missing FFIEC directory: {FFIEC_DIR}"
assert FDIC_DIR.exists(), f"Missing FDIC directory: {FDIC_DIR}"

print("Project root:", PROJECT_ROOT)
print("FFIEC directory:", FFIEC_DIR)
print("FDIC directory:", FDIC_DIR)


Project root: /Users/irfanhussain/Desktop/systemic-risk-project
FFIEC directory: /Users/irfanhussain/Desktop/systemic-risk-project/data/raw/ffiec
FDIC directory: /Users/irfanhussain/Desktop/systemic-risk-project/data/raw/fdic


## 2. Helper functions and FFIEC processing

These functions are defined **before** they are used.

In [16]:
def to_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def first_available(df, *columns):
    result = pd.Series(pd.NA, index=df.index, dtype="string")
    for col in columns:
        if col in df.columns:
            result = result.fillna(df[col])
    return result


def normalize_cert(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def prepare_part1(df):
    df = df.copy()
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = df["IDRSSD"].astype("string").str.strip()
    df["FDIC Certificate Number"] = df["FDIC Certificate Number"].astype("string").str.strip()
    df["Reporting Period End Date"] = pd.to_datetime(df["Reporting Period End Date"], errors="coerce")

    df["total_assets"] = to_numeric(first_available(df, "RCFD2170", "RCON2170"))
    df["total_equity"] = to_numeric(first_available(df, "RCFD3210", "RCON3210"))
    df["loans_net_unearned"] = to_numeric(first_available(df, "RCFDB528", "RCONB528"))
    df["total_nonaccrual"] = to_numeric(first_available(df, "RCFD1403", "RCON1403"))

    domestic = to_numeric(df["RCON2200"]) if "RCON2200" in df.columns else pd.Series(np.nan, index=df.index)
    foreign = to_numeric(df["RCFN2200"]) if "RCFN2200" in df.columns else pd.Series(np.nan, index=df.index)
    df["total_deposits"] = domestic.fillna(0) + foreign.fillna(0)

    keep = [
        "IDRSSD", "FDIC Certificate Number", "Reporting Period End Date",
        "Financial Institution Name", "Financial Institution State",
        "Financial Institution Filing Type", "total_assets", "total_equity",
        "loans_net_unearned", "total_nonaccrual", "total_deposits"
    ]
    return df[keep].copy()


def prepare_part2(df):
    df = df.copy()
    if len(df) > 0 and pd.isna(df.iloc[0]["IDRSSD"]):
        df = df.iloc[1:].copy()

    df["IDRSSD"] = df["IDRSSD"].astype("string").str.strip()
    df["Reporting Period End Date"] = pd.to_datetime(df["Reporting Period End Date"], errors="coerce")
    df["ytd_net_interest_income"] = to_numeric(df["RIAD4074"])
    df["ytd_net_income"] = to_numeric(df["RIAD4340"])

    return df[["IDRSSD", "Reporting Period End Date", "ytd_net_interest_income", "ytd_net_income"]].copy()


def ytd_to_quarterly(df, value_columns):
    df = df.sort_values(["IDRSSD", "Reporting Period End Date"]).copy()
    df["year"] = df["Reporting Period End Date"].dt.year
    df["quarter"] = df["Reporting Period End Date"].dt.quarter

    for col in value_columns:
        quarterly_col = col.replace("ytd_", "quarterly_")
        previous_value = df.groupby(["IDRSSD", "year"])[col].shift(1)
        previous_quarter = df.groupby(["IDRSSD", "year"])["quarter"].shift(1)
        df[quarterly_col] = np.where(
            df["quarter"] == 1,
            df[col],
            np.where(
                previous_quarter == df["quarter"] - 1,
                df[col] - previous_value,
                np.nan
            )
        )
    return df


def process_ffiec_year(year):
    part1_path = FFIEC_DIR / str(year) / "part1.txt"
    part2_path = FFIEC_DIR / str(year) / "part2.txt"

    p1_raw = pd.read_csv(part1_path, sep="\t", dtype=str, low_memory=False)
    p2_raw = pd.read_csv(part2_path, sep="\t", dtype=str, low_memory=False)

    p1 = prepare_part1(p1_raw)
    p2 = prepare_part2(p2_raw)

    if p1.duplicated(["IDRSSD", "Reporting Period End Date"]).any():
        raise ValueError(f"Duplicate bank-quarter key in {year} Part 1")
    if p2.duplicated(["IDRSSD", "Reporting Period End Date"]).any():
        raise ValueError(f"Duplicate bank-quarter key in {year} Part 2")

    df = p1.merge(
        p2,
        on=["IDRSSD", "Reporting Period End Date"],
        how="left",
        validate="one_to_one"
    )

    df = ytd_to_quarterly(df, ["ytd_net_interest_income", "ytd_net_income"])

    df["capital_ratio"] = df["total_equity"] / df["total_assets"]
    df["loan_to_asset"] = df["loans_net_unearned"] / df["total_assets"]
    df["deposit_to_asset"] = df["total_deposits"] / df["total_assets"]
    df["nonaccrual_to_loan"] = np.where(
        df["loans_net_unearned"] > 0,
        df["total_nonaccrual"] / df["loans_net_unearned"],
        np.nan
    )
    df["nonaccrual_to_asset"] = df["total_nonaccrual"] / df["total_assets"]
    df["roa"] = df["quarterly_net_income"] / df["total_assets"]
    df["nii_to_asset"] = df["quarterly_net_interest_income"] / df["total_assets"]
    df["has_nonaccrual"] = (df["total_nonaccrual"] > 0).astype("int8")

    ratio_cols = [
        "capital_ratio", "loan_to_asset", "deposit_to_asset",
        "nonaccrual_to_loan", "nonaccrual_to_asset", "roa", "nii_to_asset"
    ]
    df[ratio_cols] = df[ratio_cols].replace([np.inf, -np.inf], np.nan)
    return df.sort_values(["IDRSSD", "Reporting Period End Date"]).reset_index(drop=True)


## 3. Rebuild the bank-quarter panel

In [17]:
years = sorted(
    int(p.name) for p in FFIEC_DIR.iterdir()
    if p.is_dir() and p.name.isdigit()
)

print("FFIEC years:", years)
bank_quarter_panel = pd.concat(
    [process_ffiec_year(year) for year in years],
    ignore_index=True
)

bank_quarter_panel = (
    bank_quarter_panel
    .sort_values(["IDRSSD", "Reporting Period End Date"])
    .reset_index(drop=True)
)

print("Shape:", bank_quarter_panel.shape)
print("Date range:", bank_quarter_panel["Reporting Period End Date"].min(), "to", bank_quarter_panel["Reporting Period End Date"].max())

# Persist to disk so later notebooks do not depend on Python kernel state.
bank_quarter_panel.to_parquet(
    PROCESSED_DIR / "bank_quarter_panel.parquet",
    index=False
)
print("Saved bank_quarter_panel.parquet")


FFIEC years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Shape: (157921, 25)
Date range: 2018-03-31 00:00:00 to 2025-12-31 00:00:00
Saved bank_quarter_panel.parquet


## 4. Load and inspect FDIC failure data

In [18]:
fdic_files = sorted(FDIC_DIR.glob("*.csv"))
assert fdic_files, f"No FDIC CSV found in {FDIC_DIR}"

fdic_path = fdic_files[0]
fdic_clean = pd.read_csv(fdic_path, low_memory=False)
fdic_clean["cert"] = normalize_cert(fdic_clean["CERT"])
fdic_clean["failure_date"] = pd.to_datetime(fdic_clean["FAILDATE"], errors="coerce")

print("FDIC file:", fdic_path.name)
print("Failure records:", len(fdic_clean))
print("Failure date range:", fdic_clean["failure_date"].min(), "to", fdic_clean["failure_date"].max())


FDIC file: fdic_failures.csv
Failure records: 21
Failure date range: 2019-05-31 00:00:00 to 2026-07-17 00:00:00


## 5. Build FFIEC identity table and audit certificate ambiguity

In [19]:
ffiec_identity = bank_quarter_panel[[
    "IDRSSD", "FDIC Certificate Number", "Financial Institution Name",
    "Financial Institution State", "Reporting Period End Date"
]].copy()

ffiec_identity["IDRSSD"] = ffiec_identity["IDRSSD"].astype("string").str.strip()
ffiec_identity["cert"] = normalize_cert(ffiec_identity["FDIC Certificate Number"])

cert_id_counts = (
    ffiec_identity
    .dropna(subset=["cert", "IDRSSD"])
    .groupby("cert")["IDRSSD"]
    .nunique()
)

problem_certs = cert_id_counts[cert_id_counts > 1].index
failure_problem_certs = set(fdic_clean["cert"].dropna()) & set(problem_certs)

print("Ambiguous certificates in FFIEC:", len(problem_certs))
print("Ambiguous certificates appearing in FDIC failures:", len(failure_problem_certs))
print(failure_problem_certs)


Ambiguous certificates in FFIEC: 6
Ambiguous certificates appearing in FDIC failures: 0
set()


## 6. Match FDIC failures to the last FFIEC observation before failure

In [20]:
fdic_events = fdic_clean[[
    "CERT", "NAME", "failure_date", "RESTYPE", "RESTYPE1"
]].copy().reset_index(drop=True)
fdic_events["event_id"] = fdic_events.index
fdic_events["cert"] = normalize_cert(fdic_events["CERT"])

failure_candidates = fdic_events.merge(
    ffiec_identity, on="cert", how="left"
)

failure_candidates = failure_candidates[
    failure_candidates["Reporting Period End Date"].notna()
    & failure_candidates["failure_date"].notna()
    & (failure_candidates["Reporting Period End Date"] <= failure_candidates["failure_date"])
].copy()

failure_matches = (
    failure_candidates
    .sort_values(["event_id", "Reporting Period End Date"])
    .groupby("event_id")
    .tail(1)
    .reset_index(drop=True)
)

print("Total FDIC failures:", len(fdic_events))
print("Matched:", failure_matches["IDRSSD"].notna().sum())
print("Unmatched:", failure_matches["IDRSSD"].isna().sum())


Total FDIC failures: 21
Matched: 21
Unmatched: 0


## 7. Create failure_events

In [21]:
failure_events = failure_matches[[
    "event_id", "IDRSSD", "cert", "CERT", "NAME", "failure_date",
    "RESTYPE", "RESTYPE1", "Reporting Period End Date"
]].copy()

failure_events = failure_events.rename(
    columns={"Reporting Period End Date": "last_ffiec_observation_before_failure"}
).sort_values("failure_date").reset_index(drop=True)

display(failure_events)


,event_id,IDRSSD,cert,CERT,NAME,failure_date,RESTYPE,RESTYPE1,last_ffiec_observation_before_failure
0,0,431958,10716,10716,ENLOE STATE BANK,2019-05-31,FAILURE,PI,2019-03-31
1,1,3462449,58112,58112,LOUISA COMMUNITY BANK,2019-10-25,FAILURE,PA,2019-09-30
2,2,3461264,58317,58317,RESOLUTE BANK,2019-10-25,FAILURE,PA,2019-09-30
3,3,567905,21111,21111,CITY NATIONAL BANK OF NEW JERSEY,2019-11-01,FAILURE,PA,2019-09-30
4,4,145956,18265,18265,ERICSON STATE BANK,2020-02-14,FAILURE,PA,2019-12-31
5,5,627425,14361,14361,THE FIRST STATE BANK,2020-04-03,FAILURE,PA,2020-03-31
6,6,695134,16748,16748,FIRST CITY BANK OF FLORIDA,2020-10-16,FAILURE,PA,2020-09-30
7,7,224552,15426,15426,ALMENA STATE BANK,2020-10-23,FAILURE,PA,2020-09-30
8,8,802866,24735,24735,SILICON VALLEY BANK,2023-03-10,FAILURE,PA,2022-12-31
9,9,2942690,57053,57053,SIGNATURE BANK,2023-03-12,FAILURE,PA,2022-12-31


## 8. Create the next-4-quarter failure target

In [22]:
failure_lookup = (
    failure_events[["IDRSSD", "failure_date"]]
    .dropna(subset=["IDRSSD", "failure_date"])
    .groupby("IDRSSD", as_index=False)
    .agg(failure_date=("failure_date", "min"))
)

model_data = bank_quarter_panel.copy()
model_data["IDRSSD"] = model_data["IDRSSD"].astype("string").str.strip()
model_data["Reporting Period End Date"] = pd.to_datetime(model_data["Reporting Period End Date"], errors="coerce")

model_data = model_data.merge(
    failure_lookup, on="IDRSSD", how="left", validate="many_to_one"
)

model_data["prediction_end"] = (
    model_data["Reporting Period End Date"] + pd.DateOffset(years=1)
)

latest_failure_date = failure_events["failure_date"].max()
model_data["complete_4q_horizon"] = model_data["prediction_end"] <= latest_failure_date

future_failure = (
    model_data["failure_date"].notna()
    & (model_data["failure_date"] > model_data["Reporting Period End Date"])
    & (model_data["failure_date"] <= model_data["prediction_end"])
)

model_data["failure_next_4q"] = pd.NA
valid = model_data["complete_4q_horizon"]
model_data.loc[valid, "failure_next_4q"] = future_failure.loc[valid].astype("int8")

model_data["after_failure"] = (
    model_data["failure_date"].notna()
    & (model_data["Reporting Period End Date"] >= model_data["failure_date"])
)

ml_data = model_data[model_data["complete_4q_horizon"] & ~model_data["after_failure"]].copy()
ml_data["failure_next_4q"] = ml_data["failure_next_4q"].astype("int8")

print("Model dataset:", ml_data.shape)
print(ml_data["failure_next_4q"].value_counts())


Model dataset: (149092, 30)
failure_next_4q
0    149024
1        68
Name: count, dtype: int64


## 9. Save modeling datasets

In [23]:
ml_data.to_parquet(
    PROCESSED_DIR / "bank_failure_early_warning.parquet",
    index=False
)
failure_events.to_csv(
    PROCESSED_DIR / "failure_events.csv",
    index=False
)
print("Saved modeling datasets.")


Saved modeling datasets.


## 10. Time-based ML split

In [24]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = [
    "capital_ratio", "loan_to_asset", "deposit_to_asset",
    "nonaccrual_to_asset", "nonaccrual_to_loan", "roa", "nii_to_asset"
]
target_col = "failure_next_4q"

train_end = pd.Timestamp("2023-12-31")
val_end = pd.Timestamp("2024-12-31")

train_data = ml_data[ml_data["Reporting Period End Date"] <= train_end].copy()
val_data = ml_data[(ml_data["Reporting Period End Date"] > train_end) & (ml_data["Reporting Period End Date"] <= val_end)].copy()
test_data = ml_data[ml_data["Reporting Period End Date"] > val_end].copy()

X_train, y_train = train_data[feature_cols], train_data[target_col]
X_val, y_val = val_data[feature_cols], val_data[target_col]
X_test, y_test = test_data[feature_cols], test_data[target_col]

print("Train:", train_data.shape, "positives:", int(y_train.sum()))
print("Validation:", val_data.shape, "positives:", int(y_val.sum()))
print("Test:", test_data.shape, "positives:", int(y_test.sum()))


Train: (121764, 30) positives: 55
Validation: (18332, 30) positives: 9
Test: (8996, 30) positives: 4


## 11. Logistic Regression baseline

In [25]:
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logistic_pipeline.fit(X_train, y_train)
log_val_prob = logistic_pipeline.predict_proba(X_val)[:, 1]
log_test_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Validation ROC-AUC:", roc_auc_score(y_val, log_val_prob))
print("Validation PR-AUC:", average_precision_score(y_val, log_val_prob))
print("Test ROC-AUC:", roc_auc_score(y_test, log_test_prob))
print("Test PR-AUC:", average_precision_score(y_test, log_test_prob))


Validation ROC-AUC: 0.603982850940227
Validation PR-AUC: 0.0006383227987076557
Test ROC-AUC: 0.9592415480427046
Test PR-AUC: 0.09891358641358641


## 12. Random Forest baseline

In [26]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=10,
        class_weight="balanced", random_state=42, n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_val_prob = rf_pipeline.predict_proba(X_val)[:, 1]
rf_test_prob = rf_pipeline.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Validation ROC-AUC": [roc_auc_score(y_val, log_val_prob), roc_auc_score(y_val, rf_val_prob)],
    "Validation PR-AUC": [average_precision_score(y_val, log_val_prob), average_precision_score(y_val, rf_val_prob)],
    "Test ROC-AUC": [roc_auc_score(y_test, log_test_prob), roc_auc_score(y_test, rf_test_prob)],
    "Test PR-AUC": [average_precision_score(y_test, log_test_prob), average_precision_score(y_test, rf_test_prob)]
})

display(comparison)


,Model,Validation ROC-AUC,Validation PR-AUC,Test ROC-AUC,Test PR-AUC
0,Logistic Regression,0.603983,0.000638,0.959242,0.098914
1,Random Forest,0.628378,0.000866,0.971558,0.205178


## 13. Optional XGBoost baseline

In [27]:
try:
    import xgboost as xgb
    print("XGBoost installed:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed. Skip this section.")


XGBoost installed: 3.4.1


In [28]:
# Run only when XGBoost is installed.

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count
scale_pos_weight = negative_count / positive_count

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, objective="binary:logistic",
        eval_metric="logloss", random_state=42, n_jobs=-1
    ))
])

xgb_pipeline.fit(X_train, y_train)
xgb_val_prob = xgb_pipeline.predict_proba(X_val)[:, 1]
xgb_test_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

comparison = pd.concat([
    comparison,
    pd.DataFrame({
        "Model": ["XGBoost"],
        "Validation ROC-AUC": [roc_auc_score(y_val, xgb_val_prob)],
        "Validation PR-AUC": [average_precision_score(y_val, xgb_val_prob)],
        "Test ROC-AUC": [roc_auc_score(y_test, xgb_test_prob)],
        "Test PR-AUC": [average_precision_score(y_test, xgb_test_prob)]
    })
], ignore_index=True)

display(comparison)


,Model,Validation ROC-AUC,Validation PR-AUC,Test ROC-AUC,Test PR-AUC
0,Logistic Regression,0.603983,0.000638,0.959242,0.098914
1,Random Forest,0.628378,0.000866,0.971558,0.205178
2,XGBoost,0.786316,0.001580,0.978787,0.576316
